# Blitting in Matplotlib

Blitting is a technique for fast animation in matplotlib. Instead of redrawing the entire figure each frame, blitting:

1. Saves a copy of the background (everything except the animated artist)
2. Restores that background each frame
3. Redraws only the animated artist on top
4. Blits (copies) the result to the screen

This avoids re-rendering static elements like axes, labels, and gridlines every frame, which is much faster.

In a script you can do this manually with `copy_from_bbox`, `restore_region`, `draw_artist`, and `blit`. In Jupyter we use `FuncAnimation` with `blit=True`, which handles the same pipeline internally.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [2]:
x = np.linspace(0, 2 * np.pi, 100)

fig, ax = plt.subplots()

# animated=True tells matplotlib to only draw the artist when we
# explicitly request it (used by FuncAnimation's blit pipeline)
(ln,) = ax.plot(x, np.sin(x), animated=True)


def update(frame):
    """Update the animated artist each frame.

    With blit=True, FuncAnimation handles the blitting pipeline:
      - Saves a copy of the background (fig.canvas.copy_from_bbox)
      - Restores the background each frame (fig.canvas.restore_region)
      - Calls this function to update the artist
      - Re-renders only the returned artists (ax.draw_artist)
      - Blits the result to the screen (fig.canvas.blit)
    """
    ln.set_ydata(np.sin(x + (frame / 100) * np.pi))
    return (ln,)


ani = FuncAnimation(
    fig,
    update,
    frames=100,
    interval=10,  # milliseconds between frames
    blit=True,    # enable blitting for fast rendering
)

plt.close(fig)
HTML(ani.to_jshtml())